[Oregon Curriculum Network](http://4dsolutions.net/ocn/)<br />
[School of Tomorrow](School_of_Tomorrow.ipynb)

# A SQLite Periodic Table from Public CSV Files

As a Python teacher, I've kept a Flash application alive online. As I explained to my Facebook friends, security was imperfect and anyway I should have pipelines at the ready to recreate the application's data tables if necessary, going back to original sources.  That's what I'm doing here using public tables to regenerate the applications periodic table, stored as the Elements table in SQLite periodic_table.db (see below).

If you're here for the nuclear physics, I suggest visiting [my Notebook on radio-toxins and isotope decay](isotope_decay.ipynb).

In [1]:
import pandas as pd
import numpy as np
import sqlite3 as sql
from datetime import datetime

In [2]:
! ls *.csv

DWAbiz_2023.csv      DWAchecking_2023.csv periodictable.csv
DWAcard_2023.csv     PeriodicTableCSV.csv


Hmmm, for some reason we have Dawn Wicca & Associates data mixed in locally, probably because School of Tomorrow has roots in this Portland-based business. I'll leave them in for context but they won't be in the repo.

Then come two well-known Periodic Table csv files. To build our Flask application SQLite database, we'll need columns from both meaning: 

1. we need to [import them](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) 
2. [merge them](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html#pandas.DataFrame.merge) together on a common column
3. [export those columns](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html) we need, in SQLite format

We could also have turned to [an sql file consisting](https://github.com/barbaryska/periodic-table-sql/tree/master/elements) of SQL INSERT statements, using that to build our SQLite db file. 

So many ways to go!

Lets go with the import-to-dataframe and export route.

Here we go:

In [3]:
table_1 = pd.read_csv("periodictable.csv", header=None)
table_1.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,1,H,Hydrogen,"Greek elements hydro- and -gen, meaning 'water...",1.0,1,1.008[III][IV][V][VI],0.00008988,14.01,20.28,14.304,2.2,1400
1,2,He,Helium,"Greek hḗlios, 'sun'",18.0,1,4.002602(2)[III][V],0.0001785,—[VII],4.22,5.193,–,0.008
2,3,Li,Lithium,"Greek líthos, 'stone'",1.0,2,6.94[III][IV][V][VIII][VI],0.534,453.69,1560,3.582,0.98,20
3,4,Be,Beryllium,"beryl, a mineral (ultimately from the name of ...",2.0,2,9.0121831(5),1.85,1560,2742,1.825,1.57,2.8
4,5,B,Boron,"borax, a mineral (from Arabic bawraq)",13.0,2,10.81[III][IV][V][VI],2.34,2349,4200,1.026,2.04,10


After looking at all the columns, we can use additional arguments to get only those we'll be needing or just want to keep for some reason (Etymology is fun).

In [4]:
table_1 = pd.read_csv("periodictable.csv", usecols = [0, 1, 2, 3], names = ["Protons", "Abbrev", "Element", "Etymology"])

In [5]:
table_1.head()

,Protons,Abbrev,Element,Etymology
0,1,H,Hydrogen,"Greek elements hydro- and -gen, meaning 'water..."
1,2,He,Helium,"Greek hḗlios, 'sun'"
2,3,Li,Lithium,"Greek líthos, 'stone'"
3,4,Be,Beryllium,"beryl, a mineral (ultimately from the name of ..."
4,5,B,Boron,"borax, a mineral (from Arabic bawraq)"


I've already looked at `PeriodicTableCSV.csv` (it's here in the repo), so I already know what columns I want and how to name them.

In [6]:
table_2 = pd.read_csv("PeriodicTableCSV.csv", header=0, names = ["Name", "Mass", "Series", "Protons"], usecols = [0, 2, 4, 10])

In [7]:
table_2.head()

,Name,Mass,Series,Protons
0,Hydrogen,1.008000,diatomic nonmetal,1
1,Helium,4.002602,noble gas,2
2,Lithium,6.940000,alkali metal,3
3,Beryllium,9.012183,alkaline earth metal,4
4,Boron,10.810000,metalloid,5


I would've thought the column I call Abbrev (for Abbreviation) would be in both tables, but it's not. 

So now lets merge table_1.Protons to table_2.Protons and extract Element, Proton, Mass and Series. 

The Element column should include the one- or two-letter abbreviation in parentheses.

In [8]:
table_3 = table_1.merge(table_2, how="inner", on="Protons")
table_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118 entries, 0 to 117
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Protons    118 non-null    int64  
 1   Abbrev     118 non-null    object 
 2   Element    118 non-null    object 
 3   Etymology  118 non-null    object 
 4   Name       118 non-null    object 
 5   Mass       118 non-null    float64
 6   Series     118 non-null    object 
dtypes: float64(1), int64(1), object(5)
memory usage: 6.6+ KB


In [9]:
table_3.head()

,Protons,Abbrev,Element,Etymology,Name,Mass,Series
0,1,H,Hydrogen,"Greek elements hydro- and -gen, meaning 'water...",Hydrogen,1.008000,diatomic nonmetal
1,2,He,Helium,"Greek hḗlios, 'sun'",Helium,4.002602,noble gas
2,3,Li,Lithium,"Greek líthos, 'stone'",Lithium,6.940000,alkali metal
3,4,Be,Beryllium,"beryl, a mineral (ultimately from the name of ...",Beryllium,9.012183,alkaline earth metal
4,5,B,Boron,"borax, a mineral (from Arabic bawraq)",Boron,10.810000,metalloid


Rewrite the Element column with the abbreviation in parens after the name...

Here's what the sql table Elements looks like, inside of `periodic_table.db`. I'm using `sqlite3`, a command line utility, to access the database in this way. I plan to duplicate this schema, including with my initials KTU at the end of each record. The tiny Flask application that uses this table implies keeping track of record authorship and datetime last touched.

<pre>
(base) 4dsolutions:School_of_Tomorrow kirbyurner$ sqlite3 periodic_table.db
SQLite version 3.37.0 2021-12-09 01:34:53
Enter ".help" for usage hints.
sqlite> .schema Elements
CREATE TABLE Elements
            (elem_protons int PRIMARY KEY,
             elem_symbol text,
             elem_long_name text,
             elem_mass float,
             elem_series text,
             updated_at int,
             updated_by text);
</pre>

We'll need to add those to missing columns to have exporting recreate this schema, but with today's datetime (in seconds since the epoch started).

<pre>
sqlite> select * from Elements;
17|Cl|Chlorine|35.453|Halogen|1469802789|KTU
8|O|Oxygen|15.9994|Other nonmetal|1469802789|KTU
5|B|Boron|10.811|Metalloid|1469802789|KTU
15|P|Phosphorous|30.973762|Other nonmetal|1469802789|KTU
10|Ne|Neon|20.1797|Noble gas|1469802789|KTU
19|K|Potassium|39.0983|Alkali metal|1469802789|KTU
11|Na|Sodium|22.98976928|Alkali metal|1469802789|KTU
4|Be|Beryllium|9.012182|Alkaline earth metal|1469802789|KTU
18|Ar|Argon|39.948|Nobel gas|1469802789|KTU
14|Si|Silicon|28.0855|Metalloid|1768032675|KTU
3|Li|Lithium|6.941|Alkali metal|1469802789|KTU
12|Mg|Magnesium|24.305|Alkaline earth metal|1469802789|KTU
16|S|Sulfur|32.065|Other nonmetal|1469802789|KTU
</pre>

In [10]:
int(datetime.now().timestamp())

1775579385

In [11]:
table_3["Timestamp"] = int(datetime.now().timestamp())

In [12]:
table_3["Initials"] = "KTU"

Let's now rename our columns to match those of the target table, which is already defined (in terms of schema) and used in a Flask application. We need to match existing column names. We're refreshing an existing table with new data.

In [13]:
table_3 = table_3.rename(columns={"Protons":"elem_protons", "Abbrev":"elem_symbol", 
                        "Element":"elem_long_name", "Mass":"elem_mass", "Series":"elem_series",
                        "Timestamp":"updated_at", "Initials":"updated_by"})

In [14]:
table_3 = table_3.drop(columns="Etymology")

In [15]:
conn = sql.connect("periodic_table.db")

In [16]:
table_3.to_sql("Elements", conn, if_exists="replace", index=False)

118

In [17]:
? conn

Call signature:  conn(*args, **kwargs)
Type:           Connection
String form:    <sqlite3.Connection object at 0x10d058f40>
File:           ~/opt/anaconda3/envs/py311/lib/python3.11/sqlite3/__init__.py
Docstring:      SQLite database connection object.

In [18]:
# dir(conn)

In [19]:
conn.commit()

In [20]:
conn.close()